# Scan-Spike – Eval-Notebook (M2)

Misst die Erfolgskriterien des **Scan-Validierungs-Spikes** (Brain →
`vault/50_Umsetzung/Scan-Validierungs-Spike.md`) pro Testraum:
Walkthrough-Video + Ground Truth → Metrik-Tabelle + Go/Anpassen/Pivot.

**Umgebung:** Google Colab mit Gratis-GPU (`Laufzeit → Laufzeittyp ändern → T4 GPU`).
Phasen: P0 Posen · P1 Layout · P2 Tiefe · P3 Objekte · P4 Spiegel/Glas · P5 Laufzeit.

⚠️ Lizenz-Regeln (`LICENSES.md`): Depth Anything V2 nur **Small** (Apache);
Grounding DINO + SAM2 (Apache) sind der Hauptpfad, **kein YOLO/AGPL** einbauen.

In [ ]:
# Setup (Colab): Repo holen + Abhängigkeiten
# In Colab: privates Repo via Token klonen oder Videos nach testdata/ hochladen.
import sys, subprocess, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless", "transformers", "torch", "accelerate",
                    "matplotlib", "pandas"], check=True)
laufzeiten: dict[str, float] = {}

In [ ]:
# Konfiguration je Testraum
from pathlib import Path

RAUM = "r1"  # r1 = Bad (hart) · r2 = Küche · r3 = Wohnraum (leicht)
VIDEOS = sorted(Path(f"testdata/{RAUM}").glob("*.mov"))
GROUND_TRUTH = Path(f"ground-truth/{RAUM}.json")
FRAME_FPS = 2          # extrahierte Frames pro Sekunde
BLUR_SCHWELLE = 60.0   # Frames darunter werden verworfen (capture_check.py)
# Relevante Objektklassen je Raumtyp (Konfig laut Notebook-Spezifikation)
KLASSEN = {
    "r1": ["toilet", "sink", "shower", "bathtub", "mirror", "door", "window"],
    "r2": ["sink", "oven", "refrigerator", "dishwasher", "cooktop", "door", "window"],
    "r3": ["sofa", "table", "chair", "shelf", "tv", "door", "window"],
}[RAUM]
print(VIDEOS, KLASSEN)

## P0/P5 – Frames extrahieren + Qualitäts-Gate

Schärfe-Gate gemäss `capture_check.py`: Laplacian-Varianz < 60 ⇒ Frame
verwerfen. **Achtung Interpretation:** bei texturlosen weissen Wänden ist der
Wert auch bei scharfem Bild tief – das Gate sortiert also Frames aus, die für
Tiefe/Detektion ohnehin nichts hergeben (keine Features), nicht nur unscharfe.

In [ ]:
import cv2, numpy as np, time
t0 = time.time()
frames: list[np.ndarray] = []
verworfen = 0
for video in VIDEOS:
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    schritt = max(1, round(fps / FRAME_FPS))
    idx = 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if idx % schritt == 0:
            ok, frame = cap.retrieve()
            if ok:
                grau = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                if cv2.Laplacian(grau, cv2.CV_64F).var() >= BLUR_SCHWELLE:
                    frames.append(frame)
                else:
                    verworfen += 1
        idx += 1
    cap.release()
laufzeiten["frames"] = time.time() - t0
print(f"{len(frames)} Frames behalten, {verworfen} verworfen (Gate {BLUR_SCHWELLE})")

## P2 – Metrische Tiefe (Depth Anything V2 **Small**, Apache-2.0)

In [ ]:
t0 = time.time()
from transformers import pipeline
from PIL import Image
tiefe_pipe = pipeline("depth-estimation",
                      model="depth-anything/Depth-Anything-V2-Small-hf",
                      device=0 if IN_COLAB else -1)
tiefenkarten = []
for f in frames[:: max(1, len(frames) // 20)]:  # max ~20 Frames fürs Eval
    img = Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    tiefenkarten.append(np.array(tiefe_pipe(img)["depth"]))
laufzeiten["tiefe"] = time.time() - t0
print(f"{len(tiefenkarten)} Tiefenkarten in {laufzeiten['tiefe']:.0f}s")

## P3 – Objekte erkennen (Grounding DINO, offenes Vokabular, Apache-2.0)

SAM2-Instanz-Trennung ergänzen, sobald die Detektion sitzt
(`facebook/sam2-hiera-small`, Apache).

In [ ]:
t0 = time.time()
from transformers import pipeline as hf_pipeline
detektor = hf_pipeline("zero-shot-object-detection",
                       model="IDEA-Research/grounding-dino-tiny",
                       device=0 if IN_COLAB else -1)
detektionen = []
for f in frames[:: max(1, len(frames) // 20)]:
    img = Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    dets = detektor(img, candidate_labels=[f"a {k}" for k in KLASSEN], threshold=0.3)
    detektionen.append(dets)
laufzeiten["objekte"] = time.time() - t0
gefunden = sorted({d["label"] for ds in detektionen for d in ds})
print("Gefundene Klassen:", gefunden)

## P1/P4 – Layout & Spiegel/Glas (TODO nach erstem GPU-Lauf)

- **P1 Layout:** Ebenen-Fit aus Tiefe + Posen (ohne AR-Posen: Manhattan-Annahme,
  Boden-/Wand-Ebenen via RANSAC auf rückprojizierten Punkten). Variante
  «Ecken-Antippen»: manuell gesetzte Eckpunkte ersetzen geschätzte Ecken –
  beide Varianten getrennt messen!
- **P4 Spiegel:** prüfen, ob Tiefe im Spiegelbereich «hinter die Wand» springt
  (Phantomgeometrie) → Maskieren via Detektion `mirror` + Prior Wandebene.

## Metriken gegen Ground Truth (ground-truth/rX.json)

In [ ]:
import json, pandas as pd
gt = json.loads(GROUND_TRUTH.read_text(encoding="utf-8")) if GROUND_TRUTH.exists() else None
report = {
    "raum": RAUM,
    "wandlaengen_fehler_median_cm": None,   # nach P1
    "wandlaengen_fehler_max_cm": None,
    "rechtwinkel_abweichung_grad": None,
    "raumflaechen_fehler_pct": None,
    "objekt_recall_pct": None,              # nach P3 vs. gt["objekte"]
    "label_korrekt_pct": None,
    "aufnahmedauer_s": sum(json.loads(Path(f"testdata/{RAUM}/capture-check.json")
                                      .read_text())[i]["dauer_s"]
                           for i in range(len(VIDEOS))) if Path(f"testdata/{RAUM}/capture-check.json").exists() else None,
    **{f"laufzeit_{k}_s": round(v, 1) for k, v in laufzeiten.items()},
}
if gt and detektionen:
    soll = {o["klasse"] for o in gt["objekte"]}
    ist = {d["label"].removeprefix("a ").strip() for ds in detektionen for d in ds}
    report["objekt_recall_pct"] = round(100 * len(soll & ist) / len(soll), 1) if soll else None
pd.DataFrame([report]).T

## Ergebnis-Zeile in reports/ schreiben + Gate-Empfehlung

Erfolgskriterien (Spike): Wandlängen ≤8/15 cm (≤5/10 mit Ecken-Antippen) ·
Winkel ≤5° · Fläche ≤8 % · Recall ≥90 % · Label ≥80 % · Aufnahme ≤3–5 min ·
Nachbearbeitung ≤~2 min. → **Go / Go bedingt / Anpassen / Pivot** begründen und
als Learning ins Brain zurückspielen.

In [ ]:
out = Path(f"reports/{RAUM}-metriken.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("geschrieben:", out)